# Week 2 – Assignment 2
## Machine Learning Concepts + Practical on Hospital Dataset

**Name:** Kisa Fatema Haider Sayed  
**Enrollment No:** 03404092025  
**College:** IGDTUW  
**Course:** ML and Generative AI with Python

## Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import io, warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             accuracy_score, mean_squared_error, r2_score)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)

## Load Hospital Dataset

In [ ]:
# The CSV has each row wrapped in quotes with comma-separated values inside —
# we clean it before parsing.
lines = open('hospital.csv', 'r').readlines()
cleaned = [l.strip().strip('"').replace('\r', '') for l in lines]
df = pd.read_csv(io.StringIO('\n'.join(cleaned)))

print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

In [ ]:
print("Data Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nBasic Statistics:")
df.describe()

---
## Q1. Loan Approval – Classification or Regression?

**Answer:**

- This is a **Classification** problem because the output is a discrete category: *Approved* or *Rejected* (not a continuous number).
- **Most Suitable Algorithm:** **Decision Tree**
  - Linear Regression predicts continuous values, so it is unsuitable here.
  - KNN can classify but is sensitive to irrelevant features common in financial data.
  - A Decision Tree naturally handles categorical decisions (approve/reject) with interpretable rules, making it the best fit for loan approval logic.

**Practical Demonstration on Hospital Dataset:**  
We create a binary target `high_charges` (1 if charges > median, 0 otherwise) and train a Decision Tree classifier.

In [ ]:
# Create binary classification target
median_charges = df['charges'].median()
df['high_charges'] = (df['charges'] > median_charges).astype(int)

print(f"Median charges threshold: ${median_charges:,.2f}")
print("Class distribution:")
print(df['high_charges'].value_counts())

In [ ]:
# Encode categorical columns
le = LabelEncoder()
df_enc = df.copy()
for col in ['gender', 'smoker', 'region']:
    df_enc[col] = le.fit_transform(df_enc[col])

features_cls = ['age', 'gender', 'bmi', 'children', 'smoker', 'region']
X_cls = df_enc[features_cls]
y_cls = df_enc['high_charges']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42)

dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train_c, y_train_c)
y_pred_c = dt.predict(X_test_c)

print("Decision Tree Classifier Accuracy:", round(accuracy_score(y_test_c, y_pred_c) * 100, 2), "%")

---
## Q2. Feature vs Label

**Answer:**

| Term | Definition | Example (Hospital Dataset) |
|------|-----------|---------------------------|
| **Feature** | Input variables used to make a prediction | `age`, `bmi`, `smoker`, `region`, `children` |
| **Label** | The output/target variable the model tries to predict | `charges` (medical cost) |

**Real-world analogy:**  
In a house price prediction model — the *size*, *number of rooms*, and *location* are **features**; the *selling price* is the **label**.

In [ ]:
print("Features (X) in Hospital Dataset:")
print(['age', 'gender', 'bmi', 'children', 'smoker', 'region'])

print("\nLabel (y) in Hospital Dataset:")
print("charges  →  the medical cost to be predicted")

print("\nSample feature values:")
df[['age', 'bmi', 'smoker', 'charges']].head()

---
## Q3. How a Decision Tree Makes a Prediction

**Answer:**

A Decision Tree splits data by asking a series of yes/no questions on features, like a flowchart:

1. **Root Node:** Start with the most informative feature (chosen by highest *Information Gain* or lowest *Gini Impurity*).
2. **Branching:** At each node, the data is split based on a threshold (e.g., `bmi > 30?`).
3. **Leaf Node:** Once a stopping condition is met (max depth, pure node), the majority class becomes the prediction.

**Example path for "will a customer purchase?":**
> Is age > 35? → Yes → Is income > 50k? → Yes → **Predict: Purchase**

Below we print the actual tree rules learned from the hospital dataset:

In [ ]:
tree_rules = export_text(dt, feature_names=features_cls)
print("Decision Tree Rules (max_depth=4):")
print(tree_rules[:2000])   # show first portion for readability

---
## Q4. Why is KNN a Supervised Learning Algorithm?

**Answer:**

- KNN is **supervised** because it requires **labelled training data**. It stores all (feature, label) pairs and classifies a new point by looking at the labels of its K nearest neighbors.
- **Without labels**, KNN cannot assign a class — it would have no reference to vote from. In that case it becomes a *distance-based clustering* approach (unsupervised), not classification.

> **KNN cannot perform classification if labels are unavailable.** Labels are essential for the majority-vote mechanism that produces the prediction.

In [ ]:
# Demonstrating KNN on hospital dataset
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_c, y_train_c)
knn_pred = knn.predict(X_test_c)

print("KNN (K=5) Classifier Accuracy:", round(accuracy_score(y_test_c, knn_pred) * 100, 2), "%")
print("KNN uses labels from training data to vote — without them, no prediction is possible.")

---
## Q5. KNN Prediction with K=5

**Given:** K = 5, Neighbors = [Yes, Yes, No, Yes, No]

**Answer:**

| Vote | Count |
|------|-------|
| Yes  | 3     |
| No   | 2     |

**Prediction → Yes**

KNN uses **majority voting**: the class with the most votes among K neighbors wins.  
Here, *Yes* appears 3 times vs *No* 2 times → the model predicts **Yes** (customer will purchase).

In [ ]:
# Simulating the KNN vote
from collections import Counter

neighbors = ['Yes', 'Yes', 'No', 'Yes', 'No']
vote = Counter(neighbors)
prediction = vote.most_common(1)[0][0]

print("Neighbor Votes:", neighbors)
print("Vote Count:", dict(vote))
print(f"\nKNN Prediction (K=5): {prediction}")
print("Reason: 'Yes' has 3 votes vs 'No' with 2 votes → majority wins.")

---
## Q6. Linear Regression Equation: y = mx + b

**Answer:**

| Symbol | Name | Meaning |
|--------|------|---------|
| **x** | Independent variable (feature) | Input used to make prediction (e.g., `bmi`) |
| **y** | Dependent variable (label) | Output/prediction (e.g., `charges`) |
| **m** | Slope / Coefficient | How much y changes for a 1-unit increase in x |
| **b** | Intercept | Value of y when x = 0; shifts the line up or down |

**Practical:** We regress `bmi` → `charges` and extract m and b.

In [ ]:
X_lin = df[['bmi']].values
y_lin = df['charges'].values

lr = LinearRegression()
lr.fit(X_lin, y_lin)

m = lr.coef_[0]
b = lr.intercept_

print(f"Equation: charges = {m:.2f} × bmi + {b:.2f}")
print(f"\nSlope (m)     = {m:.2f}  → each 1-unit increase in BMI raises charges by ${m:.2f}")
print(f"Intercept (b) = {b:.2f}  → predicted charges when BMI = 0")

In [ ]:
# Plot the regression line
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df['bmi'], df['charges'], alpha=0.3, s=15, color='steelblue', label='Actual')
bmi_range = np.linspace(df['bmi'].min(), df['bmi'].max(), 200).reshape(-1,1)
ax.plot(bmi_range, lr.predict(bmi_range), color='tomato', lw=2, label=f'y = {m:.1f}x + {b:.0f}')
ax.set_xlabel('BMI (x)')
ax.set_ylabel('Charges / $  (y)')
ax.set_title('Linear Regression: BMI vs Medical Charges', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('linear_regression.png', dpi=150)
plt.show()

---
## Q7. What is Correlation?

**Answer:**

Correlation measures the **strength and direction of the linear relationship** between two numerical variables. It ranges from **-1 to +1**.

| Value | Meaning |
|-------|---------|
| **+1** | Perfect positive correlation — as one variable increases, the other increases proportionally |
| **0**  | No linear relationship — the variables are independent of each other |
| **-1** | Perfect negative correlation — as one increases, the other decreases proportionally |

In [ ]:
# Correlation matrix on numerical columns
num_df = df[['age', 'bmi', 'children', 'charges']]
corr = num_df.corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax, annot_kws={'size': 12})
ax.set_title('Correlation Matrix – Hospital Dataset', fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150)
plt.show()

print("Strongest correlation with 'charges':")
print(corr['charges'].drop('charges').sort_values(ascending=False))

---
## Q8. Disease Detection – Type of Error

**Given:**
| Actual Condition | Prediction |
|-----------------|------------|
| Disease Present | Healthy    |

**Answer:**

- This is a **False Negative (FN)** — the model predicted "Healthy" but the person actually has the disease.
- **Why it is dangerous:**
  - The patient does not receive timely treatment.
  - The disease may progress to a severe or life-threatening stage.
  - In contagious diseases, the patient may unknowingly spread it to others.
  - A False Negative in medical diagnosis is considered **more dangerous** than a False Positive, because missing a real disease has far worse consequences than a false alarm.

In [ ]:
# Illustrating FN with confusion matrix on our classifier
cm = confusion_matrix(y_test_c, y_pred_c)

disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Low Charges', 'High Charges'])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix – Decision Tree', fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Positives  (TP): {tp}")
print(f"True Negatives  (TN): {tn}")
print(f"False Positives (FP): {fp}  ← predicted high, actually low")
print(f"False Negatives (FN): {fn}  ← predicted low, actually high  (the dangerous case)")

---
## Q9. Confusion Matrix vs Correlation Matrix

**Answer:**

| | Confusion Matrix | Correlation Matrix |
|--|--|--|
| **Purpose** | Evaluates classification model performance | Measures linear relationship between numerical features |
| **Contents** | Counts of TP, TN, FP, FN | Correlation coefficients (−1 to +1) |
| **Used for** | Classification problems | EDA / feature selection |
| **Output** | n×n grid (n = number of classes) | n×n grid (n = number of features) |

**Confusion Matrix** tells you *how many predictions were right/wrong*.  
**Correlation Matrix** tells you *how strongly features are related to each other*.

In [ ]:
print("Confusion Matrix (Classification Evaluation):")
print(pd.DataFrame(cm,
      index=['Actual: Low', 'Actual: High'],
      columns=['Pred: Low', 'Pred: High']))

print("\nCorrelation Matrix (Feature Relationships):")
print(num_df.corr().round(2))

---
## Q10. Algorithm Comparison

| Algorithm | Core Idea |
|-----------|-----------|
| **Decision Tree** | Splits data into branches using feature-based rules (like a flowchart); easy to interpret and visualize. |
| **KNN** | Classifies a new point by majority vote among its K nearest neighbors in the feature space; no training phase. |
| **SVM** | Finds the optimal hyperplane that maximally separates classes with the widest margin; effective in high-dimensional spaces. |
| **Linear Regression** | Fits a straight line (y = mx + b) to model the relationship between a continuous target and one or more features. |
| **Neural Network** | Learns complex patterns through layers of interconnected nodes (neurons) with weights adjusted via backpropagation. |

**Practical Summary on Hospital Dataset:**

In [ ]:
# Summary comparison: Decision Tree vs KNN on hospital data
results = {
    'Decision Tree': accuracy_score(y_test_c, dt.predict(X_test_c)),
    'KNN (K=5)'    : accuracy_score(y_test_c, knn.predict(X_test_c)),
}

print("Classifier Accuracy Comparison:")
for name, acc in results.items():
    print(f"  {name:20s}: {acc*100:.2f}%")

# Linear Regression metrics
X_lr = df_enc[['age', 'bmi', 'children', 'smoker']]
y_lr = df_enc['charges']
X_tr, X_te, y_tr, y_te = train_test_split(X_lr, y_lr, test_size=0.2, random_state=42)
lr2 = LinearRegression().fit(X_tr, y_tr)
y_pr = lr2.predict(X_te)
print(f"\nLinear Regression R² Score : {r2_score(y_te, y_pr):.4f}")
print(f"Linear Regression RMSE      : ${np.sqrt(mean_squared_error(y_te, y_pr)):,.2f}")

In [ ]:
# Bar chart comparing classifier accuracies
fig, ax = plt.subplots(figsize=(7, 4))
names = list(results.keys())
accs  = [v * 100 for v in results.values()]
bars  = ax.bar(names, accs, color=['#4C72B0', '#55A868'], edgecolor='white', width=0.5)
ax.bar_label(bars, fmt='%.2f%%', padding=4, fontsize=12)
ax.set_ylim(0, 105)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Classifier Accuracy Comparison', fontweight='bold')
plt.tight_layout()
plt.savefig('algorithm_comparison.png', dpi=150)
plt.show()

---
## Summary

| Q | Topic | Key Takeaway |
|---|-------|-------------|
| Q1 | Classification vs Regression | Loan approval → Classification; Decision Tree is best |
| Q2 | Feature vs Label | Feature = input, Label = output |
| Q3 | Decision Tree logic | Splits on best feature at each node |
| Q4 | KNN & supervision | KNN needs labels; cannot classify without them |
| Q5 | KNN vote | Yes (3) > No (2) → Prediction: Yes |
| Q6 | Linear Regression | y=mx+b; m=slope, b=intercept |
| Q7 | Correlation | +1 positive, 0 none, -1 negative |
| Q8 | False Negative | Dangerous in medical diagnosis |
| Q9 | Confusion vs Correlation Matrix | Evaluation tool vs Relationship tool |
| Q10 | Algorithm comparison | Each algorithm has a unique mechanism |